In [1]:
# Import pandas/numpy, widen the display so wide tables don't get 
# truncated in output.

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

print(pd.__version__)
print(np.__version__)

3.0.5
2.5.3


In [2]:
# Load the raw Excel file into df, confirm shape is (50242, 84).
df = pd.read_excel('../data/raw/vehicles.xlsx', sheet_name='vehicles')
print(df.shape)

(50242, 84)


In [3]:
# Check dtypes/non-null counts (.info()), preview rows, 
# spot-check phevCity to see the 0-as-missing pattern.

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50242 entries, 0 to 50241
Data columns (total 84 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   barrels08        50242 non-null  float64
 1   barrelsA08       50242 non-null  float64
 2   charge120        50242 non-null  float64
 3   charge240        50242 non-null  float64
 4   city08           50242 non-null  int64  
 5   city08U          50242 non-null  float64
 6   cityA08          50242 non-null  int64  
 7   cityA08U         50242 non-null  float64
 8   cityCD           50242 non-null  float64
 9   cityE            50242 non-null  float64
 10  cityUF           50242 non-null  float64
 11  co2              50242 non-null  int64  
 12  co2A             50242 non-null  int64  
 13  co2TailpipeAGpm  50242 non-null  float64
 14  co2TailpipeGpm   50242 non-null  float64
 15  comb08           50242 non-null  int64  
 16  comb08U          50242 non-null  float64
 17  combA08          50242 

In [4]:
df.head(3)

,barrels08,barrelsA08,charge120,charge240,city08,city08U,cityA08,cityA08U,cityCD,cityE,cityUF,co2,co2A,co2TailpipeAGpm,co2TailpipeGpm,comb08,comb08U,combA08,combA08U,combE,combinedCD,combinedUF,cylinders,displ,drive,engId,eng_dscr,feScore,fuelCost08,fuelCostA08,fuelType,fuelType1,ghgScore,ghgScoreA,highway08,highway08U,highwayA08,highwayA08U,highwayCD,highwayE,highwayUF,hlv,hpv,id,lv2,lv4,make,model,mpgData,phevBlended,pv2,pv4,range,rangeCity,rangeCityA,rangeHwy,rangeHwyA,trany,UCity,UCityA,UHighway,UHighwayA,VClass,year,youSaveSpend,baseModel,guzzler,trans_dscr,tCharger,sCharger,atvType,fuelType2,rangeA,evMotor,mfrCode,c240Dscr,charge240b,c240bDscr,createdOn,modifiedOn,startStop,phevCity,phevHwy,phevComb
0,14.167143,0.0,0.0,0.0,19,0.0,0,0.0,0.0,0.0,0.0,-1,-1,0.0,423.190476,21,0.0,0,0.0,0.0,0.0,0.0,4.0,2.0,Rear-Wheel Drive,9011,(FFS),-1,2950,0,Regular,Regular Gasoline,-1,-1,25,0.0,0,0.0,0.0,0.0,0.0,0,0,1,0,0,Alfa Romeo,Spider Veloce 2000,Y,False,0,0,0,0.0,0.0,0.0,0.0,Manual 5-spd,23.3333,0.0,35.0,0.0,Two Seaters,1985,-3500,Spider,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,Tue Jan 01 00:00:00 EST 2013,Tue Jan 01 00:00:00 EST 2013,NaN,0,0,0
1,27.046364,0.0,0.0,0.0,9,0.0,0,0.0,0.0,0.0,0.0,-1,-1,0.0,807.909091,11,0.0,0,0.0,0.0,0.0,0.0,12.0,4.9,Rear-Wheel Drive,22020,(GUZZLER),-1,5650,0,Regular,Regular Gasoline,-1,-1,14,0.0,0,0.0,0.0,0.0,0.0,0,0,10,0,0,Ferrari,Testarossa,N,False,0,0,0,0.0,0.0,0.0,0.0,Manual 5-spd,11.0000,0.0,19.0,0.0,Two Seaters,1985,-17000,Testarossa,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,Tue Jan 01 00:00:00 EST 2013,Tue Jan 01 00:00:00 EST 2013,NaN,0,0,0
2,11.018889,0.0,0.0,0.0,23,0.0,0,0.0,0.0,0.0,0.0,-1,-1,0.0,329.148148,27,0.0,0,0.0,0.0,0.0,0.0,4.0,2.2,Front-Wheel Drive,2100,(FFS),-1,2300,0,Regular,Regular Gasoline,-1,-1,33,0.0,0,0.0,0.0,0.0,0.0,19,77,100,0,0,Dodge,Charger,Y,False,0,0,0,0.0,0.0,0.0,0.0,Manual 5-spd,29.0000,0.0,47.0,0.0,Subcompact Cars,1985,-250,Charger,NaN,SIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,Tue Jan 01 00:00:00 EST 2013,Tue Jan 01 00:00:00 EST 2013,NaN,0,0,0


In [5]:
df['phevCity'].value_counts().head(10)

phevCity
0     49793
29       27
24       23
34       23
50       18
37       15
27       15
26       13
33       12
21       12
Name: count, dtype: int64

In [6]:
# Narrow df down to the 25 core + electrification columns we actually need 
# (core_cols + ev_cols), .copy() to avoid future SettingWithCopyWarning.

core_cols = [
    'id', 'year', 'make', 'model', 'VClass', 'drive', 'trany',
    'cylinders', 'displ', 'fuelType', 'fuelType1', 'fuelType2', 'atvType',
    'city08', 'highway08', 'comb08', 'UCity', 'UHighway',
    'cityA08', 'highwayA08', 'combA08',
    'fuelCost08', 'co2TailpipeGpm', 'mpgData', 'startStop'
]

ev_cols = [
    'phevCity', 'phevHwy', 'phevComb',
    'range', 'rangeCity', 'rangeHwy', 'rangeA',
    'charge120', 'charge240', 'evMotor'
]

df = df[core_cols + ev_cols].copy()
print(df.shape)

(50242, 35)


In [7]:
# Fix inconsistent missing-value encodings — EV/PHEV-only columns' -1/0 → NaN; 
# retain recorded cylinders/displ = 0; do not fill missing engine fields; trany "Not Available" → NaN; 
# verify with .isna().sum().

ev_only_cols = ['phevCity', 'phevHwy', 'phevComb', 'range', 'rangeCity',
                'rangeHwy', 'rangeA', 'charge120', 'charge240']

for col in ev_only_cols:
    df[col] = df[col].replace(-1, np.nan)
    df[col] = df[col].replace(0, np.nan)

#### Engine fields
Recorded zero values are preserved. Missing values remain missing; most BEV engine fields in this snapshot are missing.

In [8]:
df['trany'] = df['trany'].replace('Not Available', np.nan)
df[ev_only_cols].isna().sum()

phevCity     49793
phevHwy      49793
phevComb     49793
range        48179
rangeCity    48856
rangeHwy     48856
rangeA       48232
charge120    50241
charge240    48243
dtype: int64

In [9]:
# Sanity-check numeric ranges on city08/highway08/comb08 
# and the year span — confirm nothing looks like a data error.

df[['city08', 'highway08', 'comb08']].describe().round(2)


,city08,highway08,comb08
count,50242.00,50242.00,50242.00
mean,21.00,26.64,23.04
std,15.42,13.00,14.24
min,6.00,9.00,7.00
25%,15.00,20.00,17.00
50%,18.00,24.00,20.00
75%,21.00,29.00,24.00
max,153.00,142.00,146.00


In [10]:

df['year'].min(), df['year'].max()

(np.int64(1984), np.int64(2027))

In [11]:
# Check for duplicate ids and fully duplicate rows — both 
# should be 0

print(df['id'].duplicated().sum())
print(df.duplicated().sum())

0


0


In [12]:
# Print a sorted missingness summary as a record of what's missing and why

missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
print(missing_summary)

charge120    50241
phevHwy      49793
phevCity     49793
phevComb     49793
rangeHwy     48856
rangeCity    48856
charge240    48243
rangeA       48232
fuelType2    48229
range        48179
evMotor      46642
atvType      43452
startStop    31689
cylinders     1619
displ         1617
drive         1186
trany           11
fuelType1        2
dtype: int64


In [13]:
# Diagnostic loop using pd.to_numeric(..., errors='coerce') across all 
# "should be numeric" columns to catch hidden string contamination — this is 
# what surfaced the rangeA slash-notation problem.

numeric_looking_cols = ['cylinders', 'displ', 'city08', 'highway08', 'comb08',
                         'UCity', 'UHighway', 'cityA08', 'highwayA08', 'combA08',
                         'fuelCost08', 'co2TailpipeGpm',
                         'phevCity', 'phevHwy', 'phevComb',
                         'range', 'rangeCity', 'rangeHwy', 'rangeA',
                         'charge120', 'charge240']

for col in numeric_looking_cols:
    coerced = pd.to_numeric(df[col], errors='coerce')
    n_broken = coerced.isna().sum() - df[col].isna().sum()
    if n_broken > 0:
        print(f"{col}: {n_broken} values couldn't convert to numeric")
        print(df.loc[coerced.isna() & df[col].notna(), col].unique()[:10])
        print()

rangeA: 417 values couldn't convert to numeric
['230/270/270' '240/290/290' '230/270' '240/280' '220/260' '220/270/260'
 '240/290/280' '250/220/320' '290/340' '270/240/340']



In [14]:
# Fix rangeA by coercing to numeric (accepting the multi-tank string values become NaN), 
# documented as a deliberate choice.

df['rangeA'] = pd.to_numeric(df['rangeA'], errors='coerce')

In [15]:
import datetime

# Fix model values incorrectly parsed as dates
mask = df['model'].map(type) == datetime.datetime

df.loc[mask, 'model'] = df.loc[mask, 'model'].apply(
    lambda x: f"{x.month}-{x.day}"
)

# Make all model values consistent text
df['model'] = df['model'].astype('string')
df['model'].map(type).value_counts()

model
<class 'str'>    50242
Name: count, dtype: int64

In [16]:
# Save the cleaned frame to data/processed/vehicles_clean.parquet.

df.to_parquet('../data/processed/vehicles_clean.parquet', index=False)
print("Saved:", df.shape)

Saved:

 (50242, 35)
